# Bengali post-OCR correction — Colab setup + dry run

Clones the repo, installs deps, sets up Tesseract + Bengali langpack, authenticates with HuggingFace, and runs a single-page correction dry run — mirrors the local environment already verified, moved here because model weights (several GB across 5 models) don't fit on the local machine's disk.

In [ ]:
!git clone https://github.com/Ariffurrhmn/bengali-postocr-sllm.git
%cd bengali-postocr-sllm

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr
!pip install -q -r requirements.txt

In [ ]:
# Bengali langpack — same tessdata_best source as the local setup (see SETUP.md)
!mkdir -p .tessdata
!curl -sL -o .tessdata/ben.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/ben.traineddata
!curl -sL -o .tessdata/eng.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/eng.traineddata
!curl -sL -o .tessdata/osd.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/osd.traineddata

In [ ]:
# HuggingFace auth — needed for the gated Llama 3.2 / Gemma 2b downloads.
# Prefer a Colab secret (key icon in the left sidebar, name it HF_TOKEN)
# over pasting the token directly into a cell.
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

## Dataset

The dataset itself (`D:\Competition_dataset_ImagesPAGEXML` locally) is not in the repo. Upload it to Colab (e.g. via Google Drive mount, or a zip upload) and point `DATASET_DIR` at it before running OCR — the frozen splits (`data/split_dev.txt`, `data/split_eval.txt`) already list which 50 page IDs are needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Set this to wherever the dataset ends up after mounting/uploading:
DATASET_DIR = '/content/drive/MyDrive/Competition_dataset_ImagesPAGEXML'

In [ ]:
import os
os.environ['TESSDATA_PREFIX'] = '/content/bengali-postocr-sllm/.tessdata'
os.environ['TESSERACT_CMD'] = 'tesseract'  # apt install puts it on PATH

%cd ocr
!python run_ocr.py --split dev
%cd ..

In [ ]:
%cd eval
!python test_metrics.py
!python score_baseline.py --split dev
%cd ..

## Correction dry run

One model, one page — sanity-check the prompt before running the full 5-model × 2-engine × 40-page eval sweep. Start with an ungated model (`titullm-1b` or `banglat5`) to avoid any HF-access surprises first.

In [ ]:
%cd correction
!python dry_run.py titullm-1b
%cd ..